In [ ]:
import numpy as np
import pandas as pd
import pickle
from utils.unsupervised import *
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import umap
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import pandas as pd
from utils.unsupervised import *

In [ ]:
!pip install openpyxl

In [ ]:
# load data
filename = "Datasets/VDJdb_PosNeg_corrected.xlsx"
cdr3_seq_column = "cdr3"
epitope_seq_column = "antigen.epitope"
final_dataset = pd.read_excel(filename, sheet_name="Sheet1")

In [ ]:
final_dataset.shape

In [ ]:
final_dataset.head(2)

In [ ]:
#onehot encoding embedding extraction

In [ ]:
# Turn the training data sequences into onehot encode embedding
import os
import pandas as pd
import gc

# Initialize results list
results_list = []
chunks_size = 1000
# Process in chunks 
for start in range(0, len(final_dataset), chunks_size):
    # Extract chunk of chunks_size rows
    df_chunk = final_dataset.iloc[start:start + chunks_size].copy()
    chunk_filename = f'cdr3_epitope_embedding_feature{start // chunks_size + 1}.csv'
    
    # Check if this chunk file already exists - if so, load it
    if os.path.exists(chunk_filename):
        existing_chunk = pd.read_csv(chunk_filename)
        # If the chunk has the same number of rows, we can assume it's fully processed
        if len(existing_chunk) == len(df_chunk):
            print(f"Skipping chunk {start // chunks_size + 1} - already processed")
            results_list.append(existing_chunk)
            continue
    
    # Process each row in the chunk
    for index, row in df_chunk.iterrows():
        cdr3_seq = row[cdr3_seq_column]
        epitope_seq = row[epitope_seq_column]
                    
            
        # Process the new data
        cdr3_embedding_features_df = build_plm_features_df(cdr3_seq, "cdr3")
        epitope_embedding_features_df = build_plm_features_df(epitope_seq, "epitope")
        
        cdr3_embedding_features_df = cdr3_embedding_features_df.reset_index(drop=True)
        # print(cdr3_embedding_features_df)
        epitope_embedding_features_df = epitope_embedding_features_df.reset_index(drop=True)
        # print(epitope_embedding_features_df)
    
        result_df = pd.concat([cdr3_embedding_features_df, epitope_embedding_features_df], axis=1)
        
        # Assign the values from result_df to the df_chunk at the correct index
        for col in result_df.columns:
            df_chunk.at[index, col] = result_df[col].iloc[0]

    
    # Append the processed chunk to results_list
    results_list.append(df_chunk)
    
    # Save the processed chunk to a file
    df_chunk.to_csv(chunk_filename, index=False)
    print(f'Saved: {chunk_filename}')
    gc.collect() 


# Combine all chunks
final_df = pd.concat(results_list, ignore_index=True)
final_df.to_csv("Datasets/VDJdb_PosNeg_corrected_onehot.csv")
print("Final dataset saved successfully")

# Create embeddings for protein sequence using ESM

In [ ]:
filename = "Datasets/VDJdb_PosNeg_corrected.xlsx"
cdr3_seq_column = "cdr3"
epitope_seq_column = "antigen.epitope"
df = pd.read_excel(filename, sheet_name="Sheet1")

In [ ]:
df.head(2)

In [ ]:
import numpy as np

unique_cdr3 = df[cdr3_seq_column].unique()
unique_epitope = df[epitope_seq_column].unique()

# Combine unique sequences from both columns
combined_unique = np.unique(np.concatenate((unique_cdr3, unique_epitope)))

# Number of unique elements across both columns:
total_unique = len(combined_unique)
print(total_unique)

In [ ]:
combined_unique

In [ ]:
import sys
import os
import pickle
from scipy import sparse
import scanpy as sc
import anndata as ad
from anndata import AnnData
import scvi
import pandas as pd
import numpy as np

import torch
import torch.nn.functional as F
import torch.nn as nn
import esm


from perturbnet.util import * 
from perturbnet.cinn.flow import * 
from perturbnet.genotypevae.genotypeVAE import *
from perturbnet.data_vae.vae import *
from perturbnet.cinn.flow_generate import SCVIZ_CheckNet2Net


%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
# load downloaded ESM model
path_ESM = ""
torch.hub.set_dir(path_ESM)
model, alphabet = esm.pretrained.esm1v_t33_650M_UR90S()

In [ ]:
# create embedding for all the sequences in the dataset
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
import glob

sequences = combined_unique
batch_size = 512
output_dir = "embedding_batches"
os.makedirs(output_dir, exist_ok=True)

batch_csv_files = []

# Process batches and save individual CSV files
for i in tqdm(range(0, len(sequences), batch_size)):
    batch_seqs = sequences[i : i + batch_size]
    # Get embeddings for the current batch
    batch_embeddings = Seq_to_Embed_ESM(batch_seqs, batch_size=len(batch_seqs),
                                       model=model, alphabet=alphabet, save_path=None)
    
    batch_embeddings_arr = np.vstack(batch_embeddings)
    
    df_batch = pd.DataFrame(batch_embeddings_arr, columns=[f'emb_dim_{j+1}' for j in range(batch_embeddings_arr.shape[1])])
    df_batch.insert(0, 'sequence', batch_seqs)
    
    csv_path = os.path.join(output_dir, f'embeddings_batch_{i//batch_size+1}.csv')
    df_batch.to_csv(csv_path, index=False)
    print(f"Saved batch {i//batch_size+1} to {csv_path}")
    batch_csv_files.append(csv_path)

# Combine all batch CSV files into a single DataFrame
combined_df = pd.concat([pd.read_csv(f) for f in batch_csv_files], ignore_index=True)

# Save combined DataFrame to a single CSV file
combined_csv_path = os.path.join(output_dir, "embeddings_combined.csv")
combined_df.to_csv(combined_csv_path, index=False)
print(f"Saved combined embeddings to {combined_csv_path}")

In [ ]:
# Generate ESM embeddings for sequences in final_df and integrate into the DataFrame

In [19]:
import pandas as pd
df1 = pd.read_csv("Datasets/VDJdb_PosNeg_corrected_onehot.csv")

In [20]:
df1

,Unnamed: 0,complex.id,gene,cdr3,v.segm,j.segm,species,mhc.a,mhc.b,mhc.class,...,epitope_embedding_M,epitope_embedding_N,epitope_embedding_P,epitope_embedding_Q,epitope_embedding_R,epitope_embedding_S,epitope_embedding_T,epitope_embedding_V,epitope_embedding_W,epitope_embedding_Y
0,0,0,TRB,CASSIVGGNEQFF,TRBV19*01,TRBJ2-1*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.111111,0.111111,0.000000,0.000000
1,1,0,TRB,CASSMRSTGELFF,TRBV19*01,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.111111,0.111111,0.000000,0.000000
2,2,0,TRB,CASSIRSAWAQYF,TRBV19*01,TRBJ2-3*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.111111,0.111111,0.000000,0.000000
3,3,0,TRB,CASSQRSTGELFF,TRBV19*01,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.111111,0.111111,0.000000,0.000000
4,4,0,TRB,CASSIRSSYEQYF,TRBV19*01,TRBJ2-7*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.111111,0.111111,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18859,18859,59974,TRB,CASSFVGAGKPNSGELFF,TRBV6-5*01,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.111111,0.000000,0.0,0.111111,0.000000,0.111111,0.111111,0.111111,0.111111,0.000000
18860,18860,39069,TRB,CASSQVGAFGELFF,TRBV6-5*01,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.111111,0.000000,0.0,0.111111,0.000000,0.111111,0.111111,0.111111,0.111111,0.000000
18861,18861,6463,TRB,CASSKGVKENTEAFF,TRBV13*01,TRBJ1-1*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.000000,0.111111,0.0,0.000000,0.111111,0.000000,0.222222,0.111111,0.000000,0.111111
18862,18862,18250,TRB,CASSQKDRAPYEQYF,TRBV4-2*01,TRBJ2-7*01,HomoSapiens,HLA-A*03:01,B2M,MHCI,...,0.000000,0.000000,0.0,0.111111,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [21]:
df2 = pd.read_csv("embedding_batches/embeddings_combined.csv")

In [22]:
df2

,sequence,emb_dim_1,emb_dim_2,emb_dim_3,emb_dim_4,emb_dim_5,emb_dim_6,emb_dim_7,emb_dim_8,emb_dim_9,...,emb_dim_1271,emb_dim_1272,emb_dim_1273,emb_dim_1274,emb_dim_1275,emb_dim_1276,emb_dim_1277,emb_dim_1278,emb_dim_1279,emb_dim_1280
0,AAFKRSCLK,-0.109915,0.061005,-0.104327,-0.132919,-0.949444,0.089251,0.135537,-0.104107,0.012651,...,-0.053235,0.208921,0.196617,-0.257186,-0.206981,-0.101246,-0.182004,-0.289396,-0.038624,0.186919
1,AAGIGILTV,-0.113724,0.013793,-0.009308,-0.033846,-0.649783,0.355944,0.072814,-0.235177,0.038957,...,-0.106239,0.137949,0.100715,-0.457425,-0.049965,-0.043273,-0.272548,-0.367148,0.111905,-0.005188
2,AALALLLLDRLNQLE,-0.077042,0.027716,-0.163542,-0.140426,-0.554431,0.108052,0.330771,-0.215910,0.044433,...,0.090943,0.201905,-0.051383,-0.320200,0.101585,-0.050468,-0.235685,-0.175652,0.114581,0.013341
3,AALPILFQV,-0.069023,0.179884,-0.058548,-0.102650,-0.640260,0.296666,0.086241,-0.143204,0.150750,...,0.112750,0.242408,-0.015785,-0.484056,-0.017952,-0.093309,-0.347610,-0.332343,0.051137,0.063321
4,AAVVRFQEAANKQKQ,0.027476,0.033787,-0.167736,-0.009103,-0.787432,0.078828,0.310062,-0.217392,-0.027171,...,0.000439,0.199491,0.172601,-0.313140,0.068028,-0.144073,-0.052653,-0.024046,0.009910,-0.024895
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18495,YVLDHLIVV,-0.096152,0.160030,-0.226271,-0.055927,-0.410878,0.210774,-0.011044,-0.173118,0.136929,...,0.115199,0.327223,-0.133655,-0.459597,-0.042710,0.056889,-0.304474,-0.230663,0.108191,-0.036675
18496,YVMAYVMAGVGS,-0.110694,0.070549,-0.146502,-0.008329,-0.491583,0.335423,0.206177,-0.307483,0.032504,...,-0.123334,0.130076,0.042395,-0.367533,-0.055279,0.120915,-0.205657,-0.286040,0.007381,0.073973
18497,YVVDDPCPI,-0.031158,0.112120,-0.113629,-0.112945,-0.602368,0.056925,-0.100441,-0.221505,0.145655,...,0.076569,0.218407,-0.011290,-0.407708,-0.044614,-0.005628,-0.363040,-0.437092,0.106882,0.010682
18498,YVWKSYVHV,-0.159756,0.242992,-0.328169,-0.095776,-0.865714,0.127688,0.049393,-0.163173,0.223714,...,0.026830,0.296392,0.175914,-0.355476,-0.095838,-0.034514,-0.180442,-0.125138,-0.002676,0.070651


In [23]:
# Assume:
# df1 is your first DataFrame with columns 'cdr3' and 'antigen.epitope'
# df2 is your second DataFrame with 'sequence' column and embedding dimension columns like 'emb_dim_1', 'emb_dim_2', ...

# Merge embeddings for 'cdr3' as 'a_emb_dim_'
df_merged = df1.merge(
    df2,
    how='left',
    left_on='cdr3',
    right_on='sequence',
    suffixes=('', '_drop_a')
)

# Rename embedding columns from df2 with 'a_emb_dim_' prefix
embedding_cols = [col for col in df2.columns if col.startswith('emb_dim_')]
rename_a = {col: 'cdr3_' + col for col in embedding_cols}
df_merged = df_merged.rename(columns=rename_a)

# Drop redundant columns from the merge
df_merged = df_merged.drop(columns=['sequence_drop_a'] if 'sequence_drop_a' in df_merged.columns else [])

# Merge embeddings for 'antigen.epitope' as 'b_emb_dim_'
df_merged = df_merged.merge(
    df2,
    how='left',
    left_on='antigen.epitope',
    right_on='sequence',
    suffixes=('', '_drop_b')
)

# Rename embedding columns from df2 with 'b_emb_dim_' prefix
rename_b = {col: 'epitope_' + col for col in embedding_cols}
df_merged = df_merged.rename(columns=rename_b)

# Drop redundant columns from this merge
df_merged = df_merged.drop(columns=['sequence_drop_b'] if 'sequence_drop_b' in df_merged.columns else [])

# Now df_merged contains all original columns plus embeddings with prefixed column names:
# For cdr3: columns like a_emb_dim_1, a_emb_dim_2, ...
# For antigen.epitope: columns like b_emb_dim_1, b_emb_dim_2, ...

# You can check:
df_merged.columns

Index(['Unnamed: 0', 'complex.id', 'gene', 'cdr3', 'v.segm', 'j.segm',
       'species', 'mhc.a', 'mhc.b', 'mhc.class',
       ...
       'epitope_emb_dim_1271', 'epitope_emb_dim_1272', 'epitope_emb_dim_1273',
       'epitope_emb_dim_1274', 'epitope_emb_dim_1275', 'epitope_emb_dim_1276',
       'epitope_emb_dim_1277', 'epitope_emb_dim_1278', 'epitope_emb_dim_1279',
       'epitope_emb_dim_1280'],
      dtype='object', length=2621)

In [24]:
df_merged

,Unnamed: 0,complex.id,gene,cdr3,v.segm,j.segm,species,mhc.a,mhc.b,mhc.class,...,epitope_emb_dim_1271,epitope_emb_dim_1272,epitope_emb_dim_1273,epitope_emb_dim_1274,epitope_emb_dim_1275,epitope_emb_dim_1276,epitope_emb_dim_1277,epitope_emb_dim_1278,epitope_emb_dim_1279,epitope_emb_dim_1280
0,0,0,TRB,CASSIVGGNEQFF,TRBV19*01,TRBJ2-1*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.005097,0.286685,-0.003035,-0.475797,-0.105294,-0.052035,-0.281661,-0.243173,0.114713,0.073654
1,1,0,TRB,CASSMRSTGELFF,TRBV19*01,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.005097,0.286685,-0.003035,-0.475797,-0.105294,-0.052035,-0.281661,-0.243173,0.114713,0.073654
2,2,0,TRB,CASSIRSAWAQYF,TRBV19*01,TRBJ2-3*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.005097,0.286685,-0.003035,-0.475797,-0.105294,-0.052035,-0.281661,-0.243173,0.114713,0.073654
3,3,0,TRB,CASSQRSTGELFF,TRBV19*01,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.005097,0.286685,-0.003035,-0.475797,-0.105294,-0.052035,-0.281661,-0.243173,0.114713,0.073654
4,4,0,TRB,CASSIRSSYEQYF,TRBV19*01,TRBJ2-7*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.005097,0.286685,-0.003035,-0.475797,-0.105294,-0.052035,-0.281661,-0.243173,0.114713,0.073654
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18859,18859,59974,TRB,CASSFVGAGKPNSGELFF,TRBV6-5*01,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.109723,0.237214,0.010498,-0.470558,-0.044414,0.010042,-0.313814,-0.320584,0.043408,0.132350
18860,18860,39069,TRB,CASSQVGAFGELFF,TRBV6-5*01,TRBJ2-2*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,0.109723,0.237214,0.010498,-0.470558,-0.044414,0.010042,-0.313814,-0.320584,0.043408,0.132350
18861,18861,6463,TRB,CASSKGVKENTEAFF,TRBV13*01,TRBJ1-1*01,HomoSapiens,HLA-A*02:01,B2M,MHCI,...,-0.169115,0.181991,0.185394,-0.240119,-0.162912,0.006917,-0.168539,-0.149826,0.010612,-0.006803
18862,18862,18250,TRB,CASSQKDRAPYEQYF,TRBV4-2*01,TRBJ2-7*01,HomoSapiens,HLA-A*03:01,B2M,MHCI,...,-0.071061,0.083227,0.157195,-0.268392,0.028861,-0.099618,-0.340728,-0.350493,0.156981,-0.017253


In [25]:
df_merged.to_csv("Datasets/VDJdb_PosNeg_corrected_onehot_esm.csv", index=False)